# Mini-projeto 1 - Fase 2: CNN para classificação do CIFAR-10

Continuidade da Fase 1 (MLP, ver `../fase1-mlp/`). A lógica reutilizável (modelo, dados, treino, métricas, checkpointing) vive no pacote `cnn_cifar10` em `../src/`, seguindo exatamente o mesmo padrão da Fase 1 — o notebook fica focado em **definir experimentos e reportar resultados**, não em implementação.

**Integrantes do grupo:** _preencher aqui (nome de todos)_

O que este notebook cobre (conforme o enunciado do mini-projeto):
- Treino de uma CNN no CIFAR-10 com hiperparâmetros configuráveis (nº/tamanho de filtros, kernel size, stride, padding, pooling, dropout, taxa de aprendizagem, além dos já cobertos na Fase 1: ativação, otimizador, função de erro).
- Métricas por classe (acurácia) e globais (acurácia, precision, recall, f1).
- Comparação direta com o melhor resultado do MLP (Fase 1: ensemble `final` = 0.6135 de acurácia) — ver `../../fase1-mlp/README.md`.
- Cada execução de treino é salva automaticamente em `../results/` (pesos + config + métricas + histórico) — ver `../src/cnn_cifar10/checkpointing.py`.

**Recomendação forte: rode este notebook com GPU** (Google Colab: Ambiente de execução > Alterar tipo de ambiente de execução > GPU; Kaggle: Settings > Accelerator > GPU T4 x2/P100 + Internet = On). CNN é bem mais lenta que MLP em CPU, e o ganho de GPU aqui é de 10-50x+. A cota gratuita de GPU do Colab pode esgotar (reseta em algumas horas) — o notebook detecta automaticamente se está no Colab ou no Kaggle, então dá pra trocar de plataforma sem editar nada além do token/secret.

## 0. Setup do ambiente

- **Local**: rode a partir de um ambiente onde o pacote já foi instalado (`pip install -e .` na pasta `fase2-cnn/`).
- **Google Colab / Kaggle Notebooks**: a célula abaixo detecta o ambiente automaticamente, clona o repositório e instala o pacote. O repositório é **privado**, então precisa de um GitHub Personal Access Token (PAT) — ver instruções abaixo, só precisa configurar uma vez por plataforma. Útil ter as duas configuradas: quando a cota de GPU de uma acabar, é só trocar para a outra sem mexer em mais nada.

### Gerar o token (uma vez só, vale para as duas plataformas)

1. No GitHub: `Settings > Developer settings > Personal access tokens > Fine-grained tokens > Generate new token`.
2. Repository access: `Only select repositories` > `redes-neurais`.
3. Permissions: `Contents` = `Read-only` (só precisa ler/clonar, não escrever).
4. Defina uma expiração (ex.: 90 dias) e gere o token — copie o valor (`github_pat_...`), ele só aparece uma vez.

### Guardar no Colab (uma vez por navegador/conta)

1. No Colab, clique no ícone de chave (🔑 **Secrets**) na barra lateral esquerda.
2. `Add new secret` → nome `GITHUB_TOKEN`, valor = o token copiado acima.
3. Ative o toggle "Notebook access" para este notebook.

### Guardar no Kaggle (uma vez por conta)

1. Abra o notebook no Kaggle, `Add-ons > Secrets`.
2. `Add Secret` → nome `GITHUB_TOKEN`, valor = o token.
3. Ainda no Kaggle: `Settings` (barra lateral direita) → `Internet` = **On** (obrigatório para clonar/instalar/baixar o dataset) e `Accelerator` = **GPU T4 x2** (ou P100).

A célula abaixo lê o secret automaticamente (Colab ou Kaggle); se não encontrar, pede o token via prompt (não fica salvo em lugar nenhum do notebook).

In [ ]:
#@title Setup (Colab, Kaggle ou local)
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB or IN_KAGGLE:
    import shutil

    BRANCH = "feat/cnn"  #@param {type:"string"}
    # Ajuste para "main" (ou o branch que estiver usando) quando o trabalho da
    # Fase 2 for mesclado - não precisa editar mais nada além desta linha.
    REPO_PATH = "github.com/jpbezerra/redes-neurais.git"
    REPO_DIR = Path("/content/redes-neurais") if IN_COLAB else Path("/kaggle/working/redes-neurais")

    # Se uma tentativa anterior de clone falhou no meio (ex.: token errado),
    # a pasta pode existir mas sem ser um repositorio git valido - nesse caso
    # apagamos e clonamos de novo em vez de só tentar "git pull" nela.
    if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
        shutil.rmtree(REPO_DIR)

    if not REPO_DIR.exists():
        token = None
        if IN_COLAB:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        elif IN_KAGGLE:
            try:
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret("GITHUB_TOKEN")
            except Exception:
                token = None
        if not token:
            import getpass
            token = getpass.getpass("Repositorio privado - cole seu GitHub Personal Access Token: ")
        clone_url = f"https://{token}@{REPO_PATH}"
        # o token fica salvo em .git/config só dentro desta VM efêmera (Colab/Kaggle)
        # - necessário para o "git pull" funcionar de novo mais tarde na mesma
        # sessão, sem pedir o token de novo.
        !git clone -q -b {BRANCH} {clone_url} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q origin {BRANCH}

    PROJECT_ROOT = REPO_DIR / "miniprojeto" / "fase2-cnn"
    assert (PROJECT_ROOT / "pyproject.toml").exists(), (
        f"Clone parece ter falhado (pyproject.toml nao encontrado em {PROJECT_ROOT}). "
        f"Confira: (1) BRANCH='{BRANCH}' e o branch certo, (2) o token em Secrets "
        "(GITHUB_TOKEN) esta valido, (3) no Kaggle, Settings > Internet = On - "
        "depois reinicie a sessao e rode esta celula de novo."
    )
    %pip install -q -e {PROJECT_ROOT}
else:
    PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> fase2-cnn/

# Garante que o pacote seja importavel nesta sessao mesmo se o "pip install -e"
# editable nao for reconhecido pelo kernel ja em execucao (comum no Colab/Kaggle:
# %pip install atualiza o site-packages, mas o processo Python atual as vezes
# so releria o sys.path apos reiniciar o runtime) - adicionar direto e mais
# robusto do que depender de reiniciar a sessao toda vez.
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
print("Ambiente:", "Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "local"))
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
#@title Imports
import torch
import matplotlib.pyplot as plt
import pandas as pd

from cnn_cifar10.config import ExperimentConfig
from cnn_cifar10.data import get_dataloaders, CLASSES
from cnn_cifar10.train import fit, fit_or_load
from cnn_cifar10.checkpointing import load_all_metadata
from cnn_cifar10.utils import set_seed, get_device

In [ ]:
#@title Device
device = get_device()
print("Usando dispositivo:", device)
if device.type == "cpu":
    print("Aviso: sem GPU disponível. CNN em CPU é bem mais lenta — considere rodar no Colab/Kaggle.")

# No Colab/Kaggle/Linux, num_workers > 0 acelera o carregamento com augmentation
# (no Windows local, exige if __name__ == '__main__': em scripts, então mantemos 0 lá).
NUM_WORKERS = 2 if (IN_COLAB or IN_KAGGLE) else 0

## 0.1 Download rápido do CIFAR-10

O servidor oficial (`cs.toronto.edu`) que o `torchvision` usa por padrão é
lento e limita a velocidade por conexão (~100 KB/s é comum, o que levaria
~30 min para baixar os ~170MB do dataset) — e em algumas redes (ex.: Kaggle)
o handshake TLS com ele falha depois do redirecionamento para
`cave.cs.toronto.edu`. A célula abaixo baixa o mesmo arquivo de um **mirror
em S3** (mantido pelo time do PyTorch para CI, mesmo conteúdo/checksum) com
**16 conexões paralelas** (`aria2c`) e extrai na pasta certa — o
`torchvision` detecta que os arquivos já existem e pula o download normal.
Se por algum motivo o download rápido falhar, não tem problema: a célula
avisa e o `torchvision` baixa do jeito normal (mais lento) na primeira vez
que `get_dataloaders` rodar. Só precisa rodar isto uma vez por sessão do
Colab/Kaggle.

In [ ]:
#@title Download rápido do CIFAR-10 (aria2c, multi-conexão)
import subprocess

# Mirror em S3 em vez do servidor oficial (cs.toronto.edu) - o oficial as
# vezes falha o handshake TLS em certas redes apos redirecionar para
# cave.cs.toronto.edu (visto no Kaggle). Mesmo arquivo/checksum.
CIFAR_URL = "https://ossci-datasets.s3.amazonaws.com/cifar-10-python.tar.gz"
cifar_extracted = DATA_DIR / "cifar-10-batches-py"
tar_path = DATA_DIR / "cifar-10-python.tar.gz"

if not cifar_extracted.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if IN_COLAB or IN_KAGGLE:
        if tar_path.exists():
            tar_path.unlink()  # remove download parcial/corrompido de uma tentativa anterior
        subprocess.run(["apt-get", "-y", "-qq", "install", "aria2"], stdout=subprocess.DEVNULL)
        result = subprocess.run([
            "aria2c", "-x", "16", "-s", "16", "-k", "1M",
            "-d", str(DATA_DIR), "-o", "cifar-10-python.tar.gz", CIFAR_URL,
        ])
        if result.returncode == 0:
            subprocess.run(["tar", "-xzf", str(tar_path), "-C", str(DATA_DIR)], check=True)
            print("CIFAR-10 baixado e extraido em", cifar_extracted)
        else:
            print("Download rapido falhou - sem problema, o torchvision baixa pelo metodo normal (mais lento) na proxima celula que chamar get_dataloaders.")
    else:
        print("Local: deixe o torchvision baixar normalmente (download=True em get_dataloaders).")
else:
    print("CIFAR-10 ja esta em", cifar_extracted)

## 1. Experimento baseline

Arquitetura equivalente à do notebook de referência do professor (`temp/CIFAR10_with_CNNs.ipynb`, adaptação do LeNet-5): 2 blocos convolucionais (32 e 64 filtros, kernel 3x3, padding 1, stride 1) + max pooling 2x2 após cada bloco, seguidos de cabeça densa `120 -> 84 -> 10`. ReLU, Adam, entropia cruzada, sem regularização — ponto de partida para a busca guiada, do mesmo jeito que a Fase 1 partiu do baseline MLP `[64,128,64]`.

In [ ]:
baseline_config = ExperimentConfig(
    run_name="baseline",
    conv_channels=(32, 64),
    kernel_size=3,
    stride=1,
    padding=1,
    pool_size=2,
    fc_layers=(120, 84),
    activation="relu",
    optimizer="adam",
    loss="cross_entropy",
    learning_rate=1e-3,
    batch_size=32,
    num_epochs=40,
    patience=5,
    notes="Baseline equivalente ao notebook de referencia (2 conv + 2 pool + 3 fc), ponto de partida da busca guiada da Fase 2.",
    tags=["baseline"],
)

set_seed(baseline_config.seed)
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=baseline_config.batch_size,
    val_fraction=baseline_config.val_fraction,
    seed=baseline_config.seed,
    num_workers=NUM_WORKERS,
    augment=baseline_config.augment,
    normalization=baseline_config.normalization,
)

result_baseline = fit_or_load(
    baseline_config, train_loader, val_loader, test_loader, device,
    class_names=list(CLASSES), results_dir=RESULTS_DIR,
)
result_baseline["test_scores"]

## 2. Leva 1 — variações isoladas (uma alavanca por vez)

Mesmo espírito da leva 1 do MLP: cada configuração muda **um** hiperparâmetro
em relação ao baseline, para medir o efeito isolado antes de combinar
vencedores em rodadas sucessivas (busca gulosa, como nas levas 2-8 do MLP).
Cobre todos os parâmetros pedidos no enunciado da Fase 2: tamanho da rede,
kernel size, stride, padding, dropout, pooling e taxa de aprendizagem — mais
batch norm e augmentation como bônus (mesma cobertura extra feita no MLP).

`num_epochs=30`/`patience=5` (menor que o baseline) para essa leva exploratória
rodar mais rápido — se algum candidato for promissor, pode retreinar depois
com mais épocas.

In [ ]:
candidate_configs = [
    ExperimentConfig(
        run_name="kernel_5",
        conv_channels=(32, 64), kernel_size=5, stride=1, padding=2, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Kernel 5x5 em vez de 3x3 (padding=2 mantem o tamanho espacial 'same').",
    ),
    ExperimentConfig(
        run_name="stride_2_no_pool",
        conv_channels=(32, 64), kernel_size=3, stride=2, padding=1, pool_size=1,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Substitui o pooling por stride=2 na propria convolucao (pool_size=1 = sem pooling extra).",
    ),
    ExperimentConfig(
        run_name="padding_0",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=0, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Convolucao 'valid' (sem padding) em vez de 'same' (padding=1) do baseline.",
    ),
    ExperimentConfig(
        run_name="pool_4",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=4,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Janela de pooling maior (4x4 em vez de 2x2) apos cada bloco convolucional.",
    ),
    ExperimentConfig(
        run_name="deeper_3conv",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Rede maior: 3 blocos convolucionais (32/64/128 filtros) em vez de 2.",
    ),
    ExperimentConfig(
        run_name="dropout_03",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), dropout=0.3, num_epochs=30, patience=5,
        notes="Dropout 0.3 (Dropout2d nos blocos conv + Dropout na cabeca densa) sobre o baseline.",
    ),
    ExperimentConfig(
        run_name="batch_norm",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), batch_norm=True, num_epochs=30, patience=5,
        notes="Adiciona BatchNorm2d apos cada convolucao, sobre o baseline.",
    ),
    ExperimentConfig(
        run_name="lr_low",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), learning_rate=5e-4, num_epochs=30, patience=5,
        notes="Learning rate 2x menor que o baseline (1e-3 -> 5e-4).",
    ),
    ExperimentConfig(
        run_name="lr_high",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), learning_rate=5e-3, num_epochs=30, patience=5,
        notes="Learning rate 5x maior que o baseline (1e-3 -> 5e-3).",
    ),
    ExperimentConfig(
        run_name="augment",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, num_epochs=30, patience=5,
        notes="Data augmentation (crop+flip) no treino, mesma transform da Fase 1.",
    ),
    # Adicione outras variacoes conforme os experimentos forem sendo decididos.
]

In [ ]:
experiment_results = {baseline_config.run_name: result_baseline}
for config in candidate_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

In [ ]:
#@title Tabela comparativa dos experimentos
comparison = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison

## 3. O que a leva 1 mostrou

Resultados (11 execuções): `augment` **0.7635** (melhor, e rodou os 30 épocas
cheios sem convergir — val_accuracy ainda subindo no fim) > `deeper_3conv`
0.7394 > `batch_norm` 0.7336 > `baseline` 0.7148 (parou em só 9 épocas,
early stopping rápido) ≈ `lr_low`/`pool_4`/`padding_0`/`dropout_03`
(neutros) > `kernel_5` 0.7035 > `stride_2_no_pool` 0.6430 > `lr_high` 0.6175
(LR alto instabiliza).

Padrão igual ao MLP: **augmentation é a alavanca isolada mais forte**, e os
"vencedores de capacidade" (batch norm, rede mais funda) overfitam rápido
sozinhos (train_loss despenca, val platô) — só devem valer combinados com
algo que regularize. `dropout=0.3` isolado não ajudou, mas pode ajudar mais
leve (`0.2`) quando combinado com augmentation (que já é uma regularização
mais fraca/lenta).

## 4. Leva 2 — combinando os vencedores

Busca gulosa (como as levas 2-8 do MLP): parte do vencedor isolado
(`augment`) e combina com os outros sinais positivos (batch norm,
profundidade, LR schedule), com mais épocas/paciência já que o treino com
augmentation converge mais devagar.

In [ ]:
round2_configs = [
    ExperimentConfig(
        run_name="augment_more_epochs",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, num_epochs=60, patience=8,
        notes="Augment sozinho rodou os 30 epochs sem convergir (val_accuracy ainda subindo) - mais epocas/paciencia para ver o teto.",
    ),
    ExperimentConfig(
        run_name="augment_batchnorm",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, num_epochs=50, patience=8,
        notes="Combina os dois melhores isolados da leva 1 (augment 0.7635 + batch_norm 0.7336).",
    ),
    ExperimentConfig(
        run_name="augment_deeper",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, num_epochs=50, patience=8,
        notes="Augment + rede mais profunda (deeper_3conv isolado deu 0.7394, mas overfitou rapido sem augment).",
    ),
    ExperimentConfig(
        run_name="augment_batchnorm_deeper",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, num_epochs=60, patience=8,
        notes="Combo dos 3 vencedores da leva 1 (augment + batch_norm + profundidade).",
    ),
    ExperimentConfig(
        run_name="augment_batchnorm_deeper_cosine",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=60, patience=8,
        notes="Combo acima + LR schedule cosine (no MLP, cosine sozinho foi neutro mas ajudou combinado com augment).",
    ),
    ExperimentConfig(
        run_name="augment_dropout_light",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, dropout=0.2, num_epochs=50, patience=8,
        notes="Dropout 0.3 isolado nao ajudou (0.7130 vs 0.7148 baseline); mais leve (0.2) combinado com augment pode regularizar sem atrapalhar a convergencia ja mais lenta.",
    ),
]

for config in round2_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

In [ ]:
#@title Tabela comparativa - leva 1 + leva 2
comparison2 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison2

## 5. O que a leva 2 mostrou

`augment_batchnorm_deeper_cosine` = **0.8548** (60 épocas) — sinergia clara:
nenhum par isolado (augment+batchnorm=0.7987, augment+deeper=0.8070) chega
perto da combinação dos três. O cosine schedule somou +1.9pp sobre o mesmo
combo sem schedule (`augment_batchnorm_deeper`=0.8357, parou por early
stopping em 0.83-0.84). Dropout leve (0.2) piorou o resultado do `augment`
sozinho (0.7450 vs 0.7635) — regularização redundante, mesmo padrão do MLP.

Olhando a curva de treino do vencedor: o `val_accuracy` achatou em
~0.85-0.856 nas últimas 8 épocas — **convergiu de verdade**, porque o
`CosineAnnealingLR` usa `T_max=num_epochs`, então o LR decai a quase zero
exatamente no fim das 60 épocas configuradas. Rodar mais épocas com o
mesmo `num_epochs=60` não ajudaria — para ver se há mais teto, a leva 3
aumenta `num_epochs` (e portanto o `T_max`) e testa outras alavancas de
capacidade/dados ao lado.

## 6. Leva 3 — esticando o vencedor da leva 2

Parte de `augment_batchnorm_deeper_cosine` (0.8548) e testa, uma de cada
vez: mais épocas/T_max, um 4º bloco convolucional, cabeça densa maior, LR
inicial mais alto (par comum com cosine annealing), augmentation mais forte
(color jitter — ao contrário do MLP, a CNN pode se beneficiar de robustez a
cor já que enxerga textura via convolução) e normalização real do CIFAR-10
(o MLP piorou com isso, mas vale reconferir com esta arquitetura).

In [ ]:
round3_configs = [
    ExperimentConfig(
        run_name="combo_cosine_long",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=90, patience=12,
        notes="Mesmo vencedor da leva 2, mas num_epochs=90 (T_max=90) para dar mais 'corda' ao cosine annealing - o de 60 epochs convergiu de verdade (val_acc achatou nas ultimas 8 epocas).",
    ),
    ExperimentConfig(
        run_name="combo_4conv",
        conv_channels=(32, 64, 128, 256), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=80, patience=10,
        notes="4o bloco convolucional (256 filtros) sobre o vencedor da leva 2 - mais capacidade de representacao.",
    ),
    ExperimentConfig(
        run_name="combo_wider_fc",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=80, patience=10,
        notes="Cabeca densa maior (256,128 em vez de 120,84) sobre o vencedor da leva 2 - tamanho da rede pelo outro eixo (fc em vez de conv).",
    ),
    ExperimentConfig(
        run_name="combo_lr_high_cosine",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=2e-3, num_epochs=80, patience=10,
        notes="LR inicial 2x maior (2e-3) - par comum com cosine annealing (comeca mais alto, decai suave ate quase zero).",
    ),
    ExperimentConfig(
        run_name="combo_strong_augment",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, augment_strength="strong", batch_norm=True,
        lr_schedule="cosine", num_epochs=80, patience=10,
        notes="Augmentation mais forte (+color jitter) sobre o vencedor - no MLP isso piorou (sem acesso a textura via convolucao), CNN pode reagir diferente.",
    ),
    ExperimentConfig(
        run_name="combo_real_norm",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        normalization="real", num_epochs=80, patience=10,
        notes="Normalizacao real do CIFAR-10 (media/desvio reais) sobre o vencedor - no MLP piorou (hipotese: batch_norm ja absorve o beneficio), reconferindo com a CNN.",
    ),
]

for config in round3_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
        augment_strength=config.augment_strength,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

In [ ]:
#@title Tabela comparativa - levas 1+2+3
comparison3 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison3

## 7. O que a leva 3 mostrou

Nenhuma das 6 variações bateu o campeão da leva 2 (`augment_batchnorm_deeper_cosine`
= 0.8548): `combo_wider_fc` 0.8527, `combo_cosine_long` 0.8522 (90 épocas,
mas a curva achatou em ~0.858-0.860 já por volta da época 60 — confirma que
o de 60 épocas já tinha convergido, mais tempo não ajuda), `combo_lr_high_cosine`
0.8516, `combo_strong_augment` 0.8489, `combo_4conv` 0.8460 (parou cedo,
época 34, por early stopping em val_loss mesmo com val_accuracy ainda
subindo — 4 blocos comprime demais o mapa espacial: 32→16→8→4→2→1),
`combo_real_norm` 0.8347 (pior — mesmo padrão do MLP: batch_norm já
absorve o benefício de normalizar a entrada).

**Conclusão:** o combo da leva 2 é um platô local forte para essa família
de arquitetura (3 blocos conv, cabeça densa pequena, augment leve, cosine).
Esticar o que já existe (mais épocas, cabeça maior, LR maior, augment mais
forte, mais profundidade) não ajuda mais. A leva 4 muda de eixo: testa
alavancas ainda não exploradas *nesse* regime (augment+batchnorm+cosine) —
largura dos filtros em vez de profundidade, regularização L2 leve, kernel
maior, dropout bem leve e batch size maior.

## 8. Leva 4 — novos eixos a partir do campeão

Mesma base do campeão da leva 2 (`conv_channels=(32,64,128)`,
`fc_layers=(120,84)`, augment leve, batch_norm, cosine, `lr=1e-3`,
`num_epochs=60`, `patience=8`), mudando uma alavanca nova por vez — ainda
não testamos largura de filtro, weight decay, kernel maior, dropout bem
leve nem batch size neste regime.

In [ ]:
round4_configs = [
    ExperimentConfig(
        run_name="combo_wider_channels",
        conv_channels=(64, 128, 256), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=60, patience=8,
        notes="Mais capacidade via largura (64/128/256 filtros) em vez de profundidade - combo_4conv (mais profundidade) piorou na leva 3, testando o outro eixo de 'tamanho da rede'.",
    ),
    ExperimentConfig(
        run_name="combo_weight_decay_light",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        weight_decay=5e-4, num_epochs=60, patience=8,
        notes="L2 leve sobre o campeao - dropout piorou (leva 2), mas weight_decay regulariza diferente (pesos, nao ativacoes) e ainda nao foi testado neste regime.",
    ),
    ExperimentConfig(
        run_name="combo_kernel5",
        conv_channels=(32, 64, 128), kernel_size=5, stride=1, padding=2, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=60, patience=8,
        notes="Kernel 5x5 (padding=2 mantem o tamanho espacial) - na leva 1 isolado foi neutro/pior, reconferindo combinado com augment+batchnorm+cosine.",
    ),
    ExperimentConfig(
        run_name="combo_dropout_tiny",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        dropout=0.1, num_epochs=60, patience=8,
        notes="Dropout bem leve (0.1) - 0.2 e 0.3 pioraram em rodadas anteriores, mas 0.1 nunca foi testado (pode ser leve o bastante para nao brigar com o augment+batchnorm).",
    ),
    ExperimentConfig(
        run_name="combo_batch64",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        batch_size=64, num_epochs=60, patience=8,
        notes="Batch size maior (64 vs 32 do padrao) - estatisticas de batch_norm mais estaveis, tambem acelera por epoca na GPU.",
    ),
]

for config in round4_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
        augment_strength=config.augment_strength,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

In [ ]:
#@title Tabela comparativa - levas 1+2+3+4
comparison4 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison4

## 9. Leva 5 — em cima do `weight_decay` (campeão da leva 4)

A leva 4 mudou o quadro: `combo_weight_decay_light` (`wd=5e-4`) fechou em
**0.8722**, +1.7pp sobre o campeão da leva 2 (`augment_batchnorm_deeper_cosine`
= 0.8548). E `combo_wider_channels` (64/128/256 filtros) marcou 0.8619 **mas
parou por early stopping na época 36 com o `val_accuracy` ainda subindo forte
(0.8804 na última época registrada)** — o critério de parada (menor `val_loss`)
cortou um modelo que ainda estava melhorando em acurácia. Dois sinais claros:
`weight_decay` é a alavanca nova vencedora, e "canais mais largos" foi
subtreinado, não ruim.

A leva 5 parte do `combo_weight_decay_light` e, uma alavanca por vez: canais
mais largos com muito mais corda, mais épocas de cosine, otimizador SGD+momentum
(receita clássica de CIFAR, nunca testada no regime de combo), L2 mais forte,
cabeça densa maior com L2 para segurar, ativação GELU, e um combo guloso das
apostas boas. `patience` bem maior nas rodadas longas para o early stopping não
repetir o corte prematuro do `combo_wider_channels`.


In [ ]:
round5_configs = [
    # base = campeão da leva 4 (combo_weight_decay_light, 0.8722):
    # conv_channels=(32,64,128), fc_layers=(120,84), augment light, batch_norm,
    # cosine, lr=1e-3, weight_decay=5e-4, num_epochs=60, patience=8
    ExperimentConfig(
        run_name="combo_wd_wider",
        conv_channels=(64, 128, 256), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=5e-4, num_epochs=100, patience=20,
        notes="Campeao da leva 4 (wd=5e-4) + canais largos (64/128/256). wider_channels sem wd (leva 4) parou na epoca 36 com val_acc ainda subindo (0.8804) - aqui com wd e MUITO mais corda (100 epocas, patience 20).",
    ),
    ExperimentConfig(
        run_name="combo_wd_long",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=5e-4, num_epochs=100, patience=20,
        notes="Campeao da leva 4 com num_epochs=100 (T_max=100). A curva dele ainda subia devagar na epoca 60 (val_acc ~0.872 -> 0.876 nas ultimas 10) - mais corda pro cosine.",
    ),
    ExperimentConfig(
        run_name="combo_wd_sgd",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=100, patience=20,
        notes="Receita classica de CIFAR: SGD+momentum 0.9, lr inicial alto (0.05) decaindo por cosine ate ~0, wd=5e-4. Costuma generalizar melhor que Adam no fim do treino. Eixo de otimizador nunca testado no regime de combo.",
    ),
    ExperimentConfig(
        run_name="combo_wd_stronger",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=1.5e-3, num_epochs=80, patience=15,
        notes="wd=5e-4 deu +1.7pp; testando 3x mais L2 (1.5e-3) pra ver se ainda ha ganho ou se ja passou do ponto.",
    ),
    ExperimentConfig(
        run_name="combo_wd_wider_fc",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=5e-4, num_epochs=80, patience=15,
        notes="Cabeca densa maior (256,128) + wd. Sem wd (leva 3) ficou neutro (0.8527); com L2 pra segurar o overfit da cabeca maior pode virar ganho.",
    ),
    ExperimentConfig(
        run_name="combo_wd_gelu",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), activation="gelu", augment=True, batch_norm=True,
        lr_schedule="cosine", learning_rate=1e-3, weight_decay=5e-4,
        num_epochs=80, patience=15,
        notes="Ativacao GELU no lugar de ReLU sobre o campeao - eixo de ativacao nunca mexido nesta fase.",
    ),
    ExperimentConfig(
        run_name="combo_wd_wider_sgd_long",
        conv_channels=(64, 128, 256), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=120, patience=25,
        notes="Combo guloso das apostas boas: canais largos (64/128/256) + SGD+cosine + wd + 120 epocas. Se wider e SGD ajudarem isolados, aqui somam.",
    ),
]

for config in round5_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
        augment_strength=config.augment_strength,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )


In [ ]:
#@title Tabela comparativa - levas 1..5
comparison5 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison5


## 10. Leva 6 — mudança de arquitetura (estilo VGG + global average pooling)

Todas as levas até aqui variaram hiperparâmetros de **uma mesma família de
arquitetura**: 1 convolução por estágio de pooling (3 convs / 3 poolings) e
cabeça densa achatada. Essa família parece ter batido no teto por volta de
0.87-0.88. Para ir além, a leva 6 mexe na arquitetura em si, usando dois campos
novos do `ExperimentConfig`:

- **`conv_layers_per_block`** — nº de convoluções empilhadas por estágio antes
  do pooling. `2` = estilo VGG (6 convs / 3 poolings): dobra a profundidade
  efetiva sem comprimir mais o mapa espacial.
- **`global_pool`** — troca o achatamento denso por `AdaptiveAvgPool2d(1)` após
  os estágios conv. Corta a maior parte dos parâmetros da cabeça (forte
  regularização) e deixa a decisão depender das features conv, não de um MLP
  grande no fim.

Base herdada da leva 5: augment leve, batch_norm, cosine, `weight_decay=5e-4`.
Rodadas longas com `patience` alto — redes maiores convergem mais devagar.


In [ ]:
round6_configs = [
    ExperimentConfig(
        run_name="vgg_2conv_per_block",
        conv_channels=(64, 128, 256), conv_layers_per_block=2,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=5e-4, num_epochs=100, patience=20,
        notes="Estilo VGG: 2 convs por estagio de pooling (6 convs / 3 poolings em vez de 3/3). A familia shallow anterior batia no teto ~0.88 - mais profundidade real e a principal alavanca que faltava.",
    ),
    ExperimentConfig(
        run_name="vgg_2conv_gap",
        conv_channels=(64, 128, 256), conv_layers_per_block=2, global_pool=True,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(128,), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=5e-4, num_epochs=100, patience=20,
        notes="VGG-ish + global average pooling: cabeca densa vira so 256->128->10. Corta a maioria dos parametros (regularizacao forte), decisao passa a depender das features conv.",
    ),
    ExperimentConfig(
        run_name="vgg_2conv_sgd",
        conv_channels=(64, 128, 256), conv_layers_per_block=2,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), augment=True, batch_norm=True, lr_schedule="cosine",
        optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=120, patience=25,
        notes="VGG-ish com a receita classica SGD+momentum+cosine - combinacao padrao pra passar de 0.90 no CIFAR com rede desse porte.",
    ),
    ExperimentConfig(
        run_name="vgg_4stage_gap_sgd",
        conv_channels=(64, 128, 256, 512), conv_layers_per_block=2, global_pool=True,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256,), augment=True, batch_norm=True, lr_schedule="cosine",
        optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=120, patience=25,
        notes="4 estagios (64/128/256/512) x2 convs + GAP: 32->16->8->4->2, o mapa nao colapsa gracas ao global pool. Maior capacidade da fase, com SGD+cosine+wd pra segurar o overfit.",
    ),
    ExperimentConfig(
        run_name="vgg_2conv_sgd_strong_aug",
        conv_channels=(64, 128, 256), conv_layers_per_block=2,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), augment=True, augment_strength="strong",
        batch_norm=True, lr_schedule="cosine",
        optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=120, patience=25,
        notes="VGG-ish + SGD + augment forte (color jitter). Na rede shallow o strong piorou (0.8489), mas rede maior aguenta - e costuma pedir - augment mais agressivo pra nao overfitar.",
    ),
    ExperimentConfig(
        run_name="vgg_2conv_dropout",
        conv_channels=(64, 128, 256), conv_layers_per_block=2,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), dropout=0.1, augment=True, batch_norm=True,
        lr_schedule="cosine", optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=120, patience=25,
        notes="VGG-ish + dropout 0.1 (Dropout2d nos estagios conv + Dropout na cabeca). Dropout leve so faz sentido com uma rede grande o bastante pra overfitar - agora ela e.",
    ),
]

for config in round6_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
        augment_strength=config.augment_strength,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )


In [ ]:
#@title Tabela comparativa - levas 1..6
comparison6 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison6


## 11. Próximas rodadas

Depois das levas 5 e 6, continue a busca gulosa: pegue o melhor `run_name`
das tabelas `comparison5`/`comparison6` e combine as alavancas vencedoras
numa leva 7 (mesmo padrão das levas 5-8 do MLP), cada rodada informada pela
anterior. Se as células ficarem grandes/lentas demais, mova para um
`scripts/run_experiments.py` (ver `../fase1-mlp/scripts/run_experiments.py`).

Alavancas de arquitetura ainda não exploradas, se a leva 6 saturar:
`conv_layers_per_block=3`, kernel 3x3 com `padding=1` + `pool_size` só nos
primeiros estágios, ou um bloco final `1x1` para misturar canais.

Vale também um **ensemble** dos melhores modelos já treinados (soft voting,
sem retreinar) — mas o objetivo declarado aqui é o **modelo único**, então
trate o ensemble como comparação extra, não como resultado principal.


In [ ]:
#@title Comparar todas as execuções salvas
df = load_all_metadata(RESULTS_DIR)
if not df.empty:
    cols = [c for c in ["run_name", "metrics.test_accuracy", "metrics.test_f1_score", "metrics.epochs_trained"] if c in df.columns]
    display(df[cols].sort_values("metrics.test_accuracy", ascending=False) if cols else df)
else:
    print("Nenhuma execucao salva ainda em", RESULTS_DIR)

## 12. Baixar `results/` para trazer de volta ao repositório local

**Só necessário no Colab/Kaggle** (local já grava direto em `../results/`).
Os resultados salvos na VM efêmera somem quando a sessão reinicia — rode
esta célula ao final de cada sessão de experimentos para não perder o
trabalho. No Colab ela gera um `.zip` e baixa direto para a pasta de
Downloads do seu computador; no Kaggle ela só gera o `.zip` em
`/kaggle/working/` — baixe pelo painel lateral **Output** do notebook
(ícone de download ao lado do arquivo). Depois é só extrair por cima da
pasta `results/` local (ou pedir para o Claude fazer isso) e dar
commit/push.

In [ ]:
#@title Baixar results/ (Colab/Kaggle)
import shutil

if IN_COLAB:
    from google.colab import files

    zip_path = shutil.make_archive("/content/results_cnn", "zip", str(RESULTS_DIR))
    print("Gerado:", zip_path)
    files.download(zip_path)
elif IN_KAGGLE:
    zip_path = shutil.make_archive("/kaggle/working/results_cnn", "zip", str(RESULTS_DIR))
    print("Gerado:", zip_path, "- baixe pelo painel lateral 'Output' do Kaggle (icone de download ao lado do arquivo).")
else:
    print("Local: results/ ja esta em", RESULTS_DIR, "- nao precisa baixar nada.")